# Apache Avro - JavaScript

All 9 JavaScript examples from [docs/avro.md](https://platob.github.io/yggdryl/avro/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## Arrow batch reads and writes

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { IOBase } = require('yggdryl')

const rows = (ids, venues) => new arrow.Table({
  id: arrow.vectorFromArray(ids.map(BigInt), new arrow.Int64()),
  venue: arrow.vectorFromArray(venues, new arrow.Utf8()),
})

const root = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-'))
const handle = new IOBase(path.join(root, 'trades.avro'))
handle.overwriteArrowTable(rows([1, 2], ['XNAS', 'XNYS']))
handle.appendArrowTable(rows([3], ['XLON']))
handle.mergeArrowTable(
  rows([2, 4], ['XPAR', null]),
  handle.recordOptions().withMergeByNames(['id']),
)

assert.equal(handle.readArrowReader().intoTable().numRows, 4)
fs.rmSync(root, { recursive: true, force: true })

### Block encoding options

In [ ]:
const assert = require('node:assert/strict')
const { RecordOptions } = require('yggdryl')

const options = RecordOptions.from('trades.avro')
  .withBlockCodec('zstandard')
  .withSyncMarker(Buffer.from('0123456789abcdef'))

assert.equal(options.blockCodec, 'zstandard')
assert.deepEqual(options.syncMarker, Buffer.from('0123456789abcdef'))

## Flexible Scalar containers and schema methods

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const schema = {
  type: 'record',
  name: 'trade',
  fields: [
    { name: 'symbol', type: 'string' },
    { name: 'quantity', type: 'long' },
  ],
}
const encoded = avro.dumps(
  [{ symbol: 'AAPL', quantity: 100 }, { symbol: 'MSFT', quantity: 25 }],
  schema,
  { source: 'docs' },
)
const decoded = avro.loads(encoded)

assert.deepEqual(decoded.metadata, { source: 'docs' })
assert.deepEqual(decoded.rows[0], { quantity: 100, symbol: 'AAPL' })

## Schemas, canonical form, and fingerprints

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const schema = new avro.Schema({
  type: 'record',
  name: 'trade',
  doc: 'one fill',
  fields: [
    { name: 'symbol', type: 'string' },
    { name: 'qty', type: 'long', 'field-id': 2 },
  ],
})

assert.ok(!schema.canonicalForm.includes('doc'))
assert.equal(Number(schema.fingerprint & 0xffn), 0xf5)
assert.equal(schema.intoJSON().fields[1]['field-id'], 2)

## Logical types decode as what they mean

In [ ]:
const assert = require('node:assert/strict')
const { Scalar, avro } = require('yggdryl')

const decimal = {
  type: 'bytes',
  logicalType: 'decimal',
  precision: 10,
  scale: 2,
}
const value = Scalar.d128(18750n, 2)
const decoded = avro.loadsSingle(avro.dumpsSingle(value, decimal), decimal)

assert.ok(decoded instanceof Scalar)
assert.equal(decoded.kind, 'd128')
assert.equal(decoded.unscaled, 18750n)
assert.equal(decoded.scale, 2)

## Reading with a different schema

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const writer = {
  type: 'record',
  name: 'trade',
  fields: [
    { name: 'symbol', type: 'string' },
    { name: 'qty', type: 'int' },
    { name: 'venue', type: 'string' },
  ],
}
const reader = new avro.Schema({
  type: 'record',
  name: 'trade',
  fields: [
    { name: 'quantity', aliases: ['qty'], type: 'long' },
    { name: 'note', type: 'string', default: 'none' },
  ],
})
const encoded = avro.dumps(
  [{ symbol: 'AAPL', qty: 100, venue: 'XNAS' }],
  writer,
)

assert.deepEqual(avro.loads(encoded, { readerSchema: reader }).rows, [
  { note: 'none', quantity: 100 },
])

## Streaming a large container

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const schema = {
  type: 'record',
  name: 'row',
  fields: [{ name: 'id', type: 'long' }],
}
const stream = avro.blocks(avro.dumps([{ id: 1 }, { id: 2 }, { id: 3 }], schema))

assert.equal(stream.schema.kind, 'record')
const block = stream.next().value
assert.equal(block.count, BigInt(block.rows().length))

## Single-object encoding

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const schema = new avro.Schema({
  type: 'record',
  name: 'tick',
  fields: [{ name: 'price', type: 'double' }],
})
const framed = avro.dumpsSingle({ price: 187.5 }, schema)

assert.deepEqual(framed.subarray(0, 2), Buffer.from([0xc3, 0x01]))
assert.deepEqual(avro.loadsSingle(framed, schema), { price: 187.5 })

## Codecs and limits

In [ ]:
const assert = require('node:assert/strict')
const { avro } = require('yggdryl')

const encoded = avro.dumps([7], '"long"')
const decoded = avro.loads(encoded, {
  maxDepth: 8,
  maxInputBytes: 1024,
  maxNodes: 8,
})

assert.deepEqual(decoded.rows, [7])